# Portfolio review copy
Original training recipe by Sabir Ali. Outputs and student identifiers are removed.
This notebook is not executed during repository preparation. See [model card](../docs/MODEL_CARD.md) for configuration inconsistencies and evaluation limits.


#  Neusoft PACS Clinical AI — Complete Fine-Tuning & RAG Pipeline
**Author:** 李如一 Ali Sabir  

##  Introduction
The main purpose of this notebook is to train a specialized Artificial Intelligence (AI) model for the **Neusoft PACS System**. It takes a general-purpose AI model (`Meta-Llama-3.1-8B-Instruct`) and teaches it how to act like a smart hospital assistant. The model is taught to handle different hospital roles (guests, patients, and doctors), guide users through booking appointments, assist doctors with AI medical image analysis (like Brain MRIs and Lung CTs), and strictly refuse to guess or make up fake medical diagnoses. Finally, the notebook packages this trained model into a format (GGUF) that can be easily run on a local computer with an RTX 5060 graphics card using Ollama.

###  Environment Setup & Dependency Installation
This cell installs the heavily optimized `unsloth` library specifically configured for Kaggle, alongside `trl` and `transformers`. It uses the `%%capture` magic command to suppress messy installation logs and verifies that all library versions are strictly compatible (specifically verifying `SFTConfig`) to prevent downstream errors during the training loop.

In [ ]:
%%capture
# Official Unsloth Kaggle install — pins ALL compatible versions together
# Do NOT split this into separate installs — version mismatch causes every error we saw
!pip install 'unsloth[kaggle-new]' --upgrade --no-cache-dir

import unsloth, trl, transformers
from trl import SFTConfig  # This import MUST succeed before continuing
print(f'unsloth      : {unsloth.__version__}')
print(f'trl          : {trl.__version__}')
print(f'transformers : {transformers.__version__}')
print()
print('INSTALL DONE — Kernel → Restart & Clear Output → then Run All')


###  GPU Initialization & VRAM Check
 Prepares the hardware environment by directing operations to the primary GPU (`CUDA_VISIBLE_DEVICES = '0'`) and configuring PyTorch memory segments to prevent Out-Of-Memory (OOM) crashes. It runs garbage collection, flushes the CUDA cache, and outputs the available VRAM to confirm the Tesla T4 GPU is ready to load the large Llama model.

In [ ]:
import os, gc
import torch

os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['PYTORCH_ALLOC_CONF']   = 'expandable_segments:True'

gc.collect()
torch.cuda.empty_cache()

free, total = torch.cuda.mem_get_info(0)
print(f'GPU  : {torch.cuda.get_device_name(0)}')
print(f'VRAM : {free/1e9:.1f} GB free / {total/1e9:.1f} GB total')

# Confirm SFTConfig is importable — proves install is correct
from trl import SFTConfig
print('SFTConfig OK — safe to continue')


### Loading Base Model & Optimized LoRA Adapters
 Instantiates the base `Meta-Llama-3.1-8B-Instruct` model in 4-bit precision to fit within the Kaggle GPU limits. It then configures and attaches the LoRA adapters to specific neural network layers (`q_proj`, `k_proj`, etc.). The parameters here are highly optimized for this domain: using `lora_alpha=32` for a stronger domain learning signal and `lora_dropout=0.05` to prevent the model from overfitting on your custom hospital dataset.

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'  # Unsloth only works on 1 GPU
os.environ['PYTORCH_ALLOC_CONF']   = 'expandable_segments:True'

import torch, gc
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 512   # reduced from 2048 — fits T4 comfortably
DTYPE          = torch.float16
LOAD_IN_4BIT   = True

gc.collect()
torch.cuda.empty_cache()

print('Loading model...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = 'unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit',
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = DTYPE,
    load_in_4bit   = LOAD_IN_4BIT,
)

# r=8 uses half the VRAM of r=16, still great for domain fine-tuning
model = FastLanguageModel.get_peft_model(
    model,
    r                          = 8,     # reduced from 16
    target_modules             = ['q_proj','k_proj','v_proj','o_proj',
                                  'gate_proj','up_proj','down_proj'],
    lora_alpha                 = 16,
    lora_dropout               = 0.05,
    bias                       = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state               = 3407,
)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
free_gb   = torch.cuda.mem_get_info(0)[0] / 1e9
print(f'  Trainable (LoRA r=8): {trainable/1e6:.2f}M ({100*trainable/total:.3f}%)')
print(f'  Free VRAM after load: {free_gb:.2f} GB')
print('✅ Model ready!')

###  Custom Neusoft Dataset Aggregation
 This cell is the data-engineering core of the notebook. It fetches large-scale medical datasets from HuggingFace and merges them with hundreds of custom-written QA pairs specific to the Neusoft System architecture. It defines exact system prompts for different "Agents" (Guest, Patient, Doctor, Refusal) to mimic the Spring Boot backend routing, finally wrapping everything into a single, cohesive HuggingFace `Dataset` object.

| Layer | Source | Count | Chatbot |
|-------|--------|-------|---------|
| A | HealthCareMagic | 1,500 | Medical empathy |
| B | MedQuAD NIH | 1,500 | Clinical accuracy |
| C | Neusoft Guest/Login page | 21 | Public chatbot |
| D | Neusoft Patient dashboard | 20 | Patient RAG chatbot |
| E | Neusoft Doctor workflows | 17 | Doctor assistant |
| F | Anti-hallucination refusals | 10 | All bots |
| G | Error & troubleshooting | 7 | All bots |


In [ ]:
from datasets import load_dataset, Dataset
import pandas as pd

# System prompts — must match your Spring Boot ChatController exactly
SYS_GUEST   = ("You are the Neusoft Hospital General Support Bot. Answer questions ONLY about "
               "registration, login, and general system information. Never invent medical diagnoses.")
SYS_PATIENT = ("You are a specialized Medical Assistant for Neusoft Hospital. "
               "Use the provided patient records to answer accurately. "
               "If the answer is not in the records, say so clearly. Never invent data.")
SYS_DOCTOR  = ("You are a clinical AI assistant for doctors at Neusoft Hospital. "
               "Help with PACS navigation and AI segmentation workflows. Be precise.")
SYS_REFUSAL = ("You are a safe hospital AI. Never fabricate diagnoses, patient data, "
               "or system features that do not exist. If unsure say: "
               "I am not certain — please consult your doctor or system administrator.")

print("Downloading external datasets...")
ds_magic   = load_dataset("wangrongsheng/HealthCareMagic-100k-en", split="train")
ds_medquad = load_dataset("AnonymousSub/MedQuAD_47441_Question_Answer_Pairs", split="train")
magic_df   = ds_magic.to_pandas().sample(1500, random_state=42)
medquad_df = ds_medquad.to_pandas().sample(1500, random_state=42)
print(f"External: {len(magic_df) + len(medquad_df)} rows")

training_data = []

for _, row in magic_df.iterrows():
    q = str(row.get("instruction","")).strip() + " " + str(row.get("input","")).strip()
    a = str(row.get("output","")).strip()
    if q.strip() and a:
        training_data.append({"messages":[
            {"role":"system","content":SYS_PATIENT},
            {"role":"user","content":q},
            {"role":"assistant","content":a}]})

for _, row in medquad_df.iterrows():
    q = str(row.get("Question","")).strip()
    a = str(row.get("Answer","")).strip()
    if q and a:
        training_data.append({"messages":[
            {"role":"system","content":SYS_DOCTOR},
            {"role":"user","content":q},
            {"role":"assistant","content":a}]})

# ── NEUSOFT GUEST / LOGIN PAGE CHATBOT ───────────────────────────────
# These match LoginRegisterView.vue -> /api/chat/ask (patientId=null)
guest_qa = [
    ("How do I register on Neusoft?",
     "Click Register on the login page. Choose Patient, Doctor, or Technician. "
     "Fill username, email, phone, password. Doctors need a License ID (MD-XXXXXX) "
     "and specialization. Patients can add Insurance ID optionally. Click Create Account."),
    ("How do I register as a doctor?",
     "Select Doctor on the register form. Fill username, password, email, phone, "
     "specialization (Brain Analysis or Lungs Analysis), and License ID (MD-XXXXXX). "
     "Your account needs admin verification before you can log in."),
    ("How do I register as a patient?",
     "Select Patient on the register form. Fill username, email, phone, password. "
     "Insurance ID is optional. You can log in immediately after registering."),
    ("How do I register as a technician?",
     "Select Technician. Fill credentials and choose your department. Admin will review."),
    ("What roles can I register as?",
     "Three roles: Patient, Doctor, and Technician. Each has its own dashboard."),
    ("How do I login?",
     "Enter your username and password on the login page and click Sign In. "
     "You will be redirected to your role-specific dashboard automatically."),
    ("I cannot login. What should I do?",
     "Check: (1) Username is case-sensitive. (2) For doctors — account may need admin verification. "
     "(3) Account may be locked by admin. (4) Wrong password. Contact hospital IT if needed."),
    ("My doctor account says Pending Verification.",
     "All doctor accounts need manual verification of your medical license by the hospital admin. "
     "This takes 1-2 business days. You can log in once the admin approves your account."),
    ("How do I reset my password?",
     "Password resets are handled by the hospital administrator. Contact IT with your username and email."),
    ("What is Neusoft PACS?",
     "Neusoft PACS is a hospital management system combining appointment management, "
     "AI medical image analysis (Brain MRI, CT Spleen, Lung CT), and digital radiology reporting."),
    ("Are you ChatGPT?",
     "No. I am the Neusoft Hospital AI Assistant running locally on hospital servers via Ollama. "
     "No data leaves the hospital network."),
    ("What can patients do?",
     "Patients can book appointments, track status (Pending/Accepted/Completed/Cancelled), "
     "view medical reports, and use the AI assistant."),
    ("What can doctors do?",
     "Doctors can accept appointments, view scans, run AI segmentation (Brain/Spleen/Lung), "
     "write and submit medical reports."),
    ("Is my data private?",
     "Yes. All data is on hospital local servers. No data is sent to external cloud services."),
    ("What file formats are supported?",
     "DICOM (.dcm or .zip), NIfTI (.nii, .nii.gz), NRRD (.nrrd). Brain MRI needs a .zip with 4 modalities."),
    ("What specializations can a doctor choose?",
     "Brain Analysis (Neuro) or Lungs Analysis (Pulmo). This determines available AI pipelines."),
    ("What is a License ID?",
     "Medical license number in MD-XXXXXX format. Required for doctor registration so admin can verify credentials."),
    ("How long does doctor verification take?",
     "1-2 business days. Admin reviews your submitted license number before activating your account."),
    ("Can I use Neusoft on my phone?",
     "Yes. Neusoft PACS is a web application accessible in any modern browser on desktop or mobile."),
    ("What AI model are you?",
     "I am the Neusoft PACS Clinical AI, built on Meta Llama 3.1 fine-tuned for Neusoft workflows. "
     "I run locally via Ollama."),
    ("Who do I contact for support?",
     "Contact your hospital IT administrator for technical issues. "
     "Use this chatbot for general questions about the Neusoft system."),
]

# ── NEUSOFT PATIENT DASHBOARD CHATBOT ────────────────────────────────
# Matches PatientDashboard.vue -> PatientChatbot.vue -> /api/chat/ask (patientId set)
patient_qa = [
    ("How do I book an appointment?",
     "Click + Book Appointment at the top right. Select a doctor, pick a date and time slot, "
     "fill in your name, then click Confirm Booking. Status starts as PENDING."),
    ("How do I see my appointments?",
     "Click My Appointments in the left sidebar. You will see all visits with status: "
     "PENDING, ACCEPTED, COMPLETED, or CANCELLED."),
    ("What does PENDING status mean?",
     "PENDING means your request was submitted and is waiting for the doctor to accept or decline. "
     "You can still cancel a PENDING appointment."),
    ("What does ACCEPTED status mean?",
     "ACCEPTED means the doctor confirmed your appointment. Please arrive at the scheduled time."),
    ("What does COMPLETED status mean?",
     "COMPLETED means the visit is done and the doctor submitted your report. "
     "Click the View Report button next to it to read your diagnosis."),
    ("What does CANCELLED status mean?",
     "CANCELLED means it was cancelled by you or declined by the doctor. You can book a new one anytime."),
    ("How do I cancel an appointment?",
     "In My Appointments, find a PENDING appointment and click Cancel. "
     "Only PENDING appointments can be cancelled this way."),
    ("How do I view my medical report?",
     "In My Appointments, find the COMPLETED appointment and click the View Report button. "
     "It shows diagnosis, doctor notes, prescription, and follow-up instructions."),
    ("When will my report be ready?",
     "Reports appear immediately after the doctor reviews your scan and submits the report. "
     "If you see COMPLETED but no View Report button, the doctor has not submitted yet."),
    ("How many appointments can I have?",
     "No limit. You can have multiple active appointments with different doctors simultaneously."),
    ("Can I change my appointment date?",
     "Cancel the PENDING appointment and book a new one with your preferred date."),
    ("What is in my medical report?",
     "Date, doctor name, department, diagnosis, clinical findings, prescription, follow-up, and doctor signature."),
    ("Is my report private?",
     "Yes. Only you and your assigned doctor can see your reports."),
    ("I see COMPLETED but no View Report button.",
     "The doctor completed the appointment but has not written the report yet. Check back later."),
    ("How do I see my reports?",
     "Click My Reports in the left sidebar. Shows all reports with date, doctor, diagnosis. "
     "Click View PDF to open the full report."),
    ("What is my patient ID?",
     "Your Patient ID is shown in the left sidebar under your username, displayed as ID: [number]."),
    ("Can I book with any doctor?",
     "Yes, you can book with any verified doctor in the system."),
    ("I have a question about my diagnosis.",
     "I can display your report information but cannot interpret diagnoses. "
     "Please contact your doctor directly or book a follow-up appointment."),
    ("Who are the available doctors?",
     "You can see all verified doctors when booking. The dropdown shows only admin-verified doctors "
     "with their specialization."),
    ("How do I log out?",
     "Click the logout arrow button at the bottom of the left sidebar."),
]

# ── DOCTOR WORKFLOWS ──────────────────────────────────────────────────
doctor_qa = [
    ("How do I accept a patient appointment?",
     "In Doctor Dashboard go to Appointments tab. Find pending requests and click Accept to confirm "
     "or Decline to reject. Accepted appointments move to your worklist."),
    ("How do I run Brain MRI AI analysis?",
     "Go to Brain Analysis in the sidebar. Select the patient, upload a .zip with all 4 MRI modalities: "
     "T1, T1CE, T2, FLAIR — all NIfTI format. Click Start Analysis. "
     "MONAI BraTS bundle runs inference and produces a 3D mesh and 2D slice previews."),
    ("What MRI files are needed for brain segmentation?",
     "All 4 modalities in one .zip: T1 (pre-contrast), T1CE (post-contrast), T2, FLAIR. "
     "All must be .nii or .nii.gz. File names must include the modality type."),
    ("How do I run CT Spleen segmentation?",
     "Go to CT Spleen / ICH in the sidebar. Upload a CT scan as .nii, .nii.gz, or DICOM .zip. "
     "Click Run Analysis. nnUNet generates a segmentation mask, 2D previews, and a 3D PLY mesh."),
    ("How do I run lung lobe segmentation?",
     "Go to AI Utilities. Upload the chest CT. Click Run. TotalSegmentator segments all 5 lung lobes "
     "and a ZIP with NRRD mask files downloads automatically."),
    ("How long does brain MRI analysis take?",
     "3-10 minutes with GPU or 20-60 minutes on CPU depending on scan size and server load."),
    ("How do I write and submit a medical report?",
     "Find the completed appointment in your worklist. Click Write Report. "
     "Fill in diagnosis, findings, prescription, and follow-up. Click Submit Report. "
     "The report immediately becomes visible to the patient."),
    ("How do I save a 2D slice image?",
     "Select your plane (Axial/Coronal/Sagittal), navigate with the slider, click Save PNG. "
     "The image downloads to your computer."),
    ("How do I save the 3D mesh as image?",
     "Position the mesh, then click Save PNG. Three.js canvas is captured as PNG. Mesh must be loaded first."),
    ("What AI models power the segmentation?",
     "Three pipelines: (1) Brain Tumor - MONAI BraTS bundle on MRI. "
     "(2) CT Spleen/ICH - nnUNet. (3) Lung Lobes - TotalSegmentator. "
     "All run on the Python server at port 5000."),
    ("What is nnUNet?",
     "nnUNet is a self-configuring medical image segmentation framework that auto-adapts to each dataset. "
     "In Neusoft PACS it powers the CT Spleen pipeline."),
    ("What is TotalSegmentator?",
     "TotalSegmentator segments 100+ anatomical structures from CT. "
     "In Neusoft it segments all 5 lung lobes."),
    ("What is the MONAI BraTS bundle?",
     "Pre-trained network from the MONAI framework trained on the BraTS challenge dataset. "
     "Segments 4 brain tumor sub-regions: necrotic core, edema, non-enhancing, and enhancing tumor."),
    ("What does the 3D mesh show?",
     "The segmented tumor region — enhancing tumor (label 4) or necrotic core (label 1). "
     "Drag to rotate, scroll to zoom, right-click to pan."),
    ("How do I view previous brain analysis cases?",
     "In Brain Analysis, select the patient from the dropdown. "
     "Previous cases appear in the history list below. Click any to reload results."),
    ("How do I view patient scans?",
     "In your worklist, find the appointment and click Open Scan or navigate to Diagnosis Studio. "
     "The CT viewer loads with axial, sagittal, and coronal views."),
    ("What is the Image Converter for?",
     "Go to Image Converter in the sidebar. Converts DICOM, NIfTI, NRRD to PNG, JPG, or NRRD. "
     "PNG/JPG extracts the middle axial slice with CT windowing."),
]

# ── ANTI-HALLUCINATION REFUSALS ───────────────────────────────────────
refusal_qa = [
    ("Can you diagnose my illness?",
     "I cannot provide medical diagnoses. Please consult a qualified doctor by booking an appointment. "
     "I can help you navigate the Neusoft system and explain your doctor-written reports."),
    ("What medication should I take?",
     "I cannot prescribe medications. Please consult your doctor. "
     "You can view existing prescriptions in your completed medical reports."),
    ("Tell me another patient's records.",
     "I cannot access other patients data. Patient records are strictly private in Neusoft PACS."),
    ("Make up a diagnosis for this scan.",
     "I cannot fabricate diagnoses. All diagnoses must be written by a licensed doctor after reviewing the scan."),
    ("What is my prognosis?",
     "Prognosis requires a specialist who can review your full clinical picture. Please discuss with your doctor."),
    ("Can you access the hospital database?",
     "No. I do not have direct database access. When you are a logged-in patient, "
     "your own records are provided as context but I cannot query other data."),
    ("Are you sure this shows cancer?",
     "I cannot confirm or deny cancer findings. Scan interpretation requires a qualified radiologist. "
     "Please refer to the medical report written by your doctor."),
    ("Write a fake medical report.",
     "I cannot generate fake reports. Medical reports must be written by licensed verified doctors."),
    ("Give me admin credentials.",
     "I cannot provide system passwords or credentials. Contact hospital IT if you have lost access."),
    ("I am emergency, diagnose me now.",
     "For a medical emergency please call emergency services immediately or go to the nearest hospital. "
     "I am an AI assistant and cannot provide emergency medical care."),
]

# ── ERROR HANDLING ────────────────────────────────────────────────────
error_qa = [
    ("Brain analysis is showing an error.",
     "Check: (1) Your zip has 4 NIfTI files with t1, t1ce, t2, flair in filenames. "
     "(2) Python AI server running on port 5000. (3) Enough disk space. "
     "Check the terminal log for the specific error."),
    ("The spleen analysis shows a processing error.",
     "Check: (1) File is .nii, .nii.gz, or DICOM .zip. (2) Python server at localhost:5000 is running. "
     "(3) spleen bundle exists at bundles/spleen_ct_segmentation/. "
     "(4) This must be a CT scan not MRI."),
    ("The 3D viewer shows blank.",
     "Try: (1) Wait for loading spinner to disappear. (2) Rotate with mouse. "
     "(3) Refresh and re-run analysis. (4) Check browser F12 console for errors."),
    ("Python backend is not connecting.",
     "Run python app7.py to start the Flask server on port 5000. "
     "Spring Boot runs separately on port 8081."),
    ("CORS error in the browser.",
     "Both Flask (flask_cors) and Spring Boot (@CrossOrigin) are configured for localhost:5173. "
     "Ensure your Vue dev server runs on port 5173."),
    ("Chatbot says AI Service Unavailable.",
     "Ollama is not running. Run ollama serve in terminal. "
     "Check ollama list to confirm neusoft-ai is loaded. API must be at localhost:11434."),
    ("How do I check if Ollama is working?",
     "Run ollama list to see models. Run ollama run neusoft-ai test to test. "
     "API endpoint: http://localhost:11434/api/generate"),
]

# Assemble all custom pairs
all_layers = [
    (guest_qa,   SYS_GUEST,   "Guest"),
    (patient_qa, SYS_PATIENT, "Patient"),
    (doctor_qa,  SYS_DOCTOR,  "Doctor"),
    (refusal_qa, SYS_REFUSAL, "Refusal"),
    (error_qa,   SYS_GUEST,   "Error"),
]

total_custom = 0
counts = {}
for qa_list, sys_p, name in all_layers:
    for q, a in qa_list:
        training_data.append({"messages":[
            {"role":"system","content":sys_p},
            {"role":"user","content":q},
            {"role":"assistant","content":a}]})
    counts[name] = len(qa_list)
    total_custom += len(qa_list)

print(f"\n{'='*50}")
print(f"  HealthCareMagic   : {len(magic_df)}")
print(f"  MedQuAD NIH       : {len(medquad_df)}")
for name, cnt in counts.items():
    print(f"  Neusoft {name:<10}: {cnt}")
print(f"  {'─'*40}")
print(f"  TOTAL             : {len(training_data)}")
print(f"{'='*50}")

final_dataset = Dataset.from_pandas(pd.DataFrame(training_data))
print("✅ Dataset ready!")


### ChatML Template Application & Tokenization
 Applies Unsloth's ChatML template formatter to the customized dataset. This cell maps the standard "system", "user", and "assistant" dictionary roles into the exact token sequence that Llama 3.1 expects. This crucial formatting step ensures the model recognizes conversational turns correctly during the Supervised Fine-Tuning phase.

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = 'chatml',
    mapping = {'role':'role','content':'content','user':'user','assistant':'assistant'}
)

def formatting_prompts_func(examples):
    texts = [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False)
             for c in examples['messages']]
    return {'text': texts}

mapped_dataset = final_dataset.map(formatting_prompts_func, batched=True)
print(f'✅ Tokenized {len(mapped_dataset)} examples')
print('\nSample (first 500 chars):')
print(mapped_dataset[0]['text'][:500])


### Supervised Fine-Tuning (SFT) Execution
Configures and initiates the Supervised Fine-Tuning process using `SFTConfig` from the `trl` library. It defines critical training hyperparameters such as running for 300 `max_steps` to ensure deep convergence and utilizing a cosine learning rate scheduler for smooth weight decay. This step actively trains the model to align with the Neusoft hospital workflows.

| Setting | Value | Reason |
|---------|-------|--------|
| `max_steps` | 300 | Full convergence |
| `lora_alpha` | 32 | 2× stronger domain signal |
| `lr_scheduler` | cosine | Smoother decay |
| `packing` | False | Avoids Unsloth packing bug |
| `report_to` | none | No wandb/hub errors |


In [ ]:
import gc, time
import torch
from trl import SFTTrainer, SFTConfig

gc.collect()
torch.cuda.empty_cache()
print(f'Free VRAM before training: {torch.cuda.mem_get_info(0)[0]/1e9:.2f} GB')


sft_config = SFTConfig(
    dataset_text_field          = 'text',
    max_seq_length              = 2048,
    dataset_num_proc            = 2,
    packing                     = False,
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps                = 10,
    max_steps                   = 300,
    learning_rate               = 2e-4,
    fp16                        = not torch.cuda.is_bf16_supported(),
    bf16                        = torch.cuda.is_bf16_supported(),
    optim                       = 'adamw_8bit',
    weight_decay                = 0.01,
    lr_scheduler_type           = 'cosine',
    logging_steps               = 5,
    seed                        = 3407,
    output_dir                  = 'neusoft_outputs',
    save_steps                  = 100,
    save_total_limit            = 2,
    report_to                   = 'none',
)

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = mapped_dataset,
    args          = sft_config,
)

print(f'Free VRAM after init: {torch.cuda.mem_get_info(0)[0]/1e9:.2f} GB')
print('Starting training...')

start_time    = time.time()
trainer_stats = trainer.train()
elapsed       = time.time() - start_time
print(f'\nDone!  Loss: {trainer_stats.training_loss:.4f}  Time: {elapsed/60:.1f} min')


## Artifact Export & Deliverables Packaging
 The final cell prepares the deliverables for your deployment and thesis. It converts the fine-tuned LoRA weights into the `.gguf` format required for Ollama. It then uses `IPython.display` to provide a convenient download link to `Neusoft_Complete.zip`, which packages the GGUF model, the Ollama Modelfile, a deployment README, and the plotted training metrics.



In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

log_history = trainer.state.log_history
train_logs  = [x for x in log_history if 'loss' in x and 'eval_loss' not in x]
steps       = [x['step'] for x in train_logs]
losses      = [x['loss']  for x in train_logs]

def ema(v, a=0.3):
    s, o = v[0], []
    for x in v:
        s = a*x+(1-a)*s; o.append(s)
    return o

smoothed     = ema(losses)
perplexities = [np.exp(min(l, 20)) for l in losses]

C = dict(bg='#0d1829', card='#111f35', text='#e2e8f0', muted='#475569',
         blue='#38bdf8', green='#4ade80', orange='#fb923c',
         purple='#c084fc', red='#f87171')

fig = plt.figure(figsize=(18,12), facecolor=C['bg'])
gs  = gridspec.GridSpec(2, 3, hspace=0.45, wspace=0.35)

def ax_s(ax, t):
    ax.set_facecolor(C['card'])
    for sp in ax.spines.values(): sp.set_color(C['muted'])
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    ax.tick_params(colors=C['text'], labelsize=8.5)
    ax.xaxis.label.set_color(C['muted']); ax.yaxis.label.set_color(C['muted'])
    ax.set_title(t, color=C['text'], fontsize=11, fontweight='bold', pad=10)
    return ax

# 1. Loss curve
ax1 = ax_s(fig.add_subplot(gs[0,0]), 'Training Loss')
ax1.plot(steps, losses,   color=C['blue'], alpha=0.25, lw=1, label='Raw')
ax1.plot(steps, smoothed, color=C['blue'], lw=2.5, label='EMA smoothed')
ax1.axhline(min(losses), color=C['green'], ls='--', lw=1)
ax1.text(steps[0], min(losses)+0.02, f'Best: {min(losses):.4f}', color=C['green'], fontsize=8)
ax1.set_xlabel('Step'); ax1.set_ylabel('Loss')
ax1.legend(facecolor=C['card'], edgecolor=C['muted'], labelcolor=C['text'], fontsize=8)

# 2. Perplexity
ax2 = ax_s(fig.add_subplot(gs[0,1]), 'Perplexity')
ax2.fill_between(steps, perplexities, alpha=0.18, color=C['purple'])
ax2.plot(steps, perplexities, color=C['purple'], lw=2)
ax2.text(0.6, 0.85, f'Final: {perplexities[-1]:.2f}', transform=ax2.transAxes,
         color=C['purple'], fontsize=9,
         bbox=dict(boxstyle='round,pad=0.4', fc=C['card'], ec=C['purple']))
ax2.set_xlabel('Step'); ax2.set_ylabel('Perplexity')

# 3. Phase means
ax3 = ax_s(fig.add_subplot(gs[0,2]), 'Loss by Phase')
n = len(losses)
phases = ['Warmup\n0-10%','Early\n10-40%','Mid\n40-70%','Late\n70-100%']
slices = [losses[:max(1,n//10)], losses[n//10:n*4//10],
          losses[n*4//10:n*7//10], losses[n*7//10:]]
means  = [np.mean(s) if s else 0 for s in slices]
bars = ax3.bar(phases, means, color=[C['red'],C['orange'],C['blue'],C['green']],
               alpha=0.85, width=0.55)
for b, v in zip(bars, means):
    ax3.text(b.get_x()+b.get_width()/2, b.get_height()+0.005,
             f'{v:.3f}', ha='center', color=C['text'], fontsize=8)
ax3.set_ylabel('Mean Loss')

# 4. Delta
ax4 = ax_s(fig.add_subplot(gs[1,0]), 'Loss Delta (Improvement Rate)')
if len(losses) > 1:
    d  = [losses[i-1]-losses[i] for i in range(1,len(losses))]
    dc = [C['green'] if x>0 else C['red'] for x in d]
    bw = max(1, steps[-1]//len(d))
    ax4.bar(steps[1:], d, color=dc, alpha=0.75, width=bw)
    ax4.axhline(0, color=C['muted'], lw=0.8)
ax4.set_xlabel('Step'); ax4.set_ylabel('Delta Loss')

# 5. Summary table
ax5 = ax_s(fig.add_subplot(gs[1,1]), 'Training Summary')
ax5.axis('off')
rows = [
    ['Base Model',      'Llama-3.1-8B'],
    ['LoRA r/alpha',    '16 / 32'],
    ['Total Steps',     str(trainer_stats.global_step)],
    ['Initial Loss',    f'{losses[0]:.4f}'],
    ['Final Loss',      f'{losses[-1]:.4f}'],
    ['Best Loss',       f'{min(losses):.4f}'],
    ['Reduction',       f'{(1-losses[-1]/losses[0])*100:.1f}%'],
    ['Perplexity',      f'{perplexities[-1]:.2f}'],
    ['Train Time',      f'{elapsed/60:.1f} min'],
    ['Total Examples',  str(len(training_data))],
    ['Custom Neusoft',  str(total_custom)],
]
tbl = ax5.table(cellText=rows, colLabels=['Metric','Value'],
                cellLoc='left', loc='center', bbox=[0,0,1,1])
tbl.auto_set_font_size(False); tbl.set_fontsize(8.5)
for (r,c), cell in tbl.get_celld().items():
    if r == 0:
        cell.set_facecolor(C['blue']); cell.set_text_props(color='#0d1829', fontweight='bold')
    else:
        cell.set_facecolor(C['bg'] if r%2==0 else C['card'])
        cell.set_text_props(color=C['text'])
    cell.set_edgecolor(C['muted'])

# 6. Convergence
ax6 = ax_s(fig.add_subplot(gs[1,2]), 'Convergence (Last 50%)')
mid = len(losses)//2
ax6.plot(steps[mid:], losses[mid:], color=C['orange'], alpha=0.35, lw=1)
ax6.plot(steps[mid:], ema(losses[mid:]), color=C['orange'], lw=2.5)
z = np.polyfit(steps[mid:], losses[mid:], 1)
ax6.plot(steps[mid:], np.poly1d(z)(steps[mid:]), color=C['red'],
         ls='--', lw=1.5, label=f'Slope: {z[0]:.5f}')
ax6.text(0.05, 0.1, 'Converging' if z[0]<0 else 'NOT converging',
         transform=ax6.transAxes,
         color=C['green'] if z[0]<0 else C['red'], fontsize=9)
ax6.legend(facecolor=C['card'], edgecolor=C['muted'], labelcolor=C['text'], fontsize=8)
ax6.set_xlabel('Step'); ax6.set_ylabel('Loss')

fig.suptitle('Neusoft PACS Clinical AI — Fine-Tuning Results',
             color=C['text'], fontsize=14, fontweight='bold', y=0.99)
plt.savefig('neusoft_training_results.png', dpi=150,
            bbox_inches='tight', facecolor=C['bg'])
plt.show()
print("Plot saved as neusoft_training_results.png")


## Pre-Export Inference Test

Tests the 4 main chatbot scenarios. Check that refusal works properly.


In [ ]:
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

tests = [
    ('How do I register as a doctor?',                       SYS_GUEST,   'Guest Bot'),
    ('My appointment is COMPLETED. How do I view my report?',SYS_PATIENT, 'Patient Bot'),
    ('Can you diagnose my illness from my symptoms?',        SYS_REFUSAL, 'Refusal Test'),
    ('How do I run brain MRI segmentation?',                 SYS_DOCTOR,  'Doctor Bot'),
]

print('='*65)
print('  NEUSOFT AI INFERENCE TEST')
print('='*65)

for question, sys_p, label in tests:
    msgs = [{'role':'system','content':sys_p}, {'role':'user','content':question}]

    # FIX: use return_tensors='pt' AND extract .input_ids to get a plain tensor
    inputs = tokenizer.apply_chat_template(
        msgs,
        tokenize            = True,
        add_generation_prompt = True,
        return_tensors      = 'pt',
        return_dict         = True,   # returns BatchEncoding with .input_ids
    ).to('cuda')

    input_ids = inputs.input_ids   # plain tensor — .shape works on this

    out = model.generate(
        input_ids          = input_ids,
        attention_mask     = inputs.attention_mask,
        max_new_tokens     = 150,
        temperature        = 0.2,
        top_p              = 0.85,
        repetition_penalty = 1.2,
        use_cache          = True,
    )

    # Slice off the prompt tokens, decode only the reply
    reply = tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True).strip()

    print(f'\n[{label}]')
    print(f'Q: {question}')
    print(f'A: {reply[:400]}')
    print('-'*65)

## GGUF Export + Modelfile for Ollama

`q4_k_m` = ~4.8 GB, ~20 tok/s on RTX 5060. The Modelfile sets `temperature=0.2` to minimize hallucination.


In [ ]:
import os, shutil

# Clean disk
print('1. Cleaning disk...')
os.system('rm -rf /kaggle/working/neusoft_outputs')
os.system('rm -f /kaggle/working/*.gguf')
os.makedirs('/kaggle/working/Neusoft_Final', exist_ok=True)

# Export GGUF to /tmp (70GB space)
print('\n2. Exporting GGUF to /tmp (~5 min)...')
model.save_pretrained_gguf('/tmp/Neusoft_Model', tokenizer, quantization_method='q4_k_m')

# Move final file
print('\n3. Moving to output...')
for src in ['/tmp/Neusoft_Model-unsloth.Q4_K_M.gguf',
            '/tmp/Neusoft_Model_gguf/Meta-Llama-3.1-8B-Instruct.Q4_K_M.gguf']:
    if os.path.exists(src):
        shutil.move(src, '/kaggle/working/Neusoft_Final/Neusoft_PACS.gguf')
        break

sz = os.path.getsize('/kaggle/working/Neusoft_Final/Neusoft_PACS.gguf')/1e9
print(f'   GGUF: {sz:.2f} GB')

# Write Modelfile — temperature=0.2 is CRITICAL for anti-hallucination
mf_lines = [
    'FROM ./Neusoft_PACS.gguf\n',
    '\n',
    'SYSTEM """You are the Neusoft PACS Clinical AI Assistant.\n',
    'You help: Guests (registration/login), Patients (appointments/reports), Doctors (AI workflows).\n',
    '\n',
    'RULES:\n',
    '1. NEVER fabricate diagnoses, patient data, or medical results.\n',
    '2. NEVER invent system features that do not exist.\n',
    '3. If unsure say: I am not certain - please consult your doctor or system administrator.\n',
    '4. Never share one patient data with another.\n',
    '5. Never prescribe medications.\n',
    '6. For emergencies: Call emergency services immediately.\n',
    '"""\n',
    '\n',
    'PARAMETER temperature 0.2\n',
    'PARAMETER top_p 0.85\n',
    'PARAMETER repeat_penalty 1.2\n',
    'PARAMETER num_ctx 4096\n',
    'PARAMETER num_predict 512\n',
    'PARAMETER stop "<|im_end|>"\n',
    'PARAMETER stop "<|eot_id|>"\n',
]
with open('/kaggle/working/Neusoft_Final/Modelfile', 'w') as f:
    f.writelines(mf_lines)
print('4. Modelfile written.')

# Write README
readme_lines = [
    '# Neusoft PACS AI - Deployment\n',
    '\n',
    '## Quick Deploy on Lenovo Legion RTX 5060\n',
    '\n',
    '1. Copy Neusoft_PACS.gguf and Modelfile to same folder\n',
    '2. ollama create neusoft-ai -f Modelfile\n',
    '3. ollama run neusoft-ai "How do I register as a doctor?"\n',
    '\n',
    'Spring Boot already configured for http://localhost:11434/api/generate\n',
    '\n',
    '## Performance RTX 5060\n',
    'Size: ~4.8 GB  Speed: ~20 tok/s  Context: 4096 tokens\n',
]
with open('/kaggle/working/Neusoft_Final/README.md', 'w') as f:
    f.writelines(readme_lines)

if os.path.exists('neusoft_training_results.png'):
    shutil.copy('neusoft_training_results.png', '/kaggle/working/Neusoft_Final/')

os.system('cd /kaggle/working && zip -r0 Neusoft_Complete.zip Neusoft_Final/')
print('\n' + '='*50)
print('  EXPORT COMPLETE!')
print('  Files in Neusoft_Final/')
print('    Neusoft_PACS.gguf  <- model')
print('    Modelfile          <- ollama create neusoft-ai -f Modelfile')
print('    README.md          <- deploy steps')
print('    training_results.png')
print('='*50)


In [ ]:
from IPython.display import FileLink, display
display(FileLink('Neusoft_Complete.zip'))
print('Download includes:')
print('  Neusoft_PACS.gguf              <- fine-tuned model for Ollama')
print('  Modelfile                      <- ollama create neusoft-ai -f Modelfile')
print('  README.md                      <- deployment steps for RTX 5060')
print('  neusoft_training_results.png   <- 6-panel training metrics for thesis')
